# Trabajo Fin de Máster  
### Análisis de la Ciudad mediante Aprendizaje Supervisado  
#### Detección Automática de Tipologías Residenciales y Patrones de Cerramiento: Interpretabilidad vs Rendimiento

**Master Universitario en Ciencia de Datos e Ingeniería de Computadores (Universidad de Granada)**

> **Autor:** David Fernández Martínez    
> **Email personal:** david.fernxndez.martinez@gmail.com  
> **Email académico:** davidfm8@correo.ugr.es  
> **LinkedIn:** [linkedin.com/in/david-fernández-martínez](https://www.linkedin.com/in/david-fern%C3%A1ndez-mart%C3%ADnez/)  
> **GitHub:** [github.com/davidfernxndez](https://github.com/davidfernxndez)

---

## Discusión para seleccionar el modelo más adecuado para el despliegue en producción

### 📝 Descripción del notebook
En los *notebooks* [3.2_Performance_results.ipynb](3.2_Performance_results.ipynb) y [4.2_Global_interp_analysis.ipynb](4.2_Global_interp_analysis.ipynb) se han evaluado, respectivamente, el rendimiento predictivo y la interpretabilidad de los modelos. En este *notebook* se integran los resultados obtenidos desde ambas perspectivas con el objetivo de establecer una valoración conjunta del equilibrio entre capacidad predictiva e interpretabilidad, determinando qué modelo, junto con su mecanismo de explicabilidad asociado, se ajusta mejor a las características del problema para su despliegue en un entorno de producción.

# 1. Análisis conjunto del rendimiento y la interpretabilidad

El principal hallazgo del estudio de interpretabilidad global es la elevada consistencia del conocimiento urbanístico obtenido mediante los distintos modelos evaluados. Tanto los modelos transparentes como los de caja negra identifican un núcleo común de variables relevantes y ofrecen una caracterización muy similar de los diferentes grados de cerramiento. Las diferencias observadas son reducidas y no conducen a interpretaciones contradictorias del fenómeno, sino que aportan matices complementarios que enriquecen su comprensión. Un ejemplo de ello es el grado de cerramiento *Simbólico*, donde *XGBoost* incorpora variables con una contribución secundaria que apenas adquiere relevancia en el resto de modelos, proporcionando una caracterización más detallada de esta categoría.

La coherencia entre el conocimiento extraído por los modelos y el conocimiento del dominio, así como la posibilidad de interpretar el comportamiento de los modelos de caja negra mediante *SHAP*, permite centrar la selección del modelo en dos aspectos fundamentales: el rendimiento predictivo y el mecanismo de explicabilidad asociado. Mientras que el primero determina la capacidad del modelo para clasificar correctamente los distintos grados de cerramiento, el segundo condiciona la forma en que sus decisiones pueden ser comprendidas y comunicadas a los expertos del dominio, requisito indispensable para su aplicación práctica.

Los mecanismos de explicabilidad empleados en este trabajo pueden agruparse en dos enfoques. El primero corresponde a la interpretación aditiva, representada por los coeficientes de la regresión logística y los valores *SHAP*, que explican las predicciones a partir de la contribución individual de cada variable. El segundo corresponde a la interpretación basada en reglas, representada por el árbol de decisión, cuyas predicciones se justifican mediante una secuencia jerárquica de condiciones lógicas que definen regiones concretas del espacio de características.

En el contexto específico de este problema, se ha identificado que las explicaciones obtenidas mediante mecanismos de interpretación aditivos presentan una mayor alineación con la forma en que los expertos del dominio comprenden las relaciones entre las variables urbanísticas y el grado de cerramiento que aquellas basadas en reglas de decisión.

Esta diferencia puede atribuirse a la naturaleza semántica de las variables, ya que cada una representa un elemento urbanístico con un significado individual reconocible y relacionado con el grado de cerramiento. En este contexto, los mecanismos aditivos permiten analizar el efecto individual de cada variable, tanto en magnitud como en dirección, y obtener una descripción conceptual del grado de cerramiento a través del conjunto de variables más influyente en su clasificación.

Por el contrario, las reglas de un árbol de decisión requieren interpretar combinaciones secuenciales de condiciones lógicas, donde la relevancia de cada variable depende del contexto definido por las condiciones previas. En el análisis realizado en [4.2_Global_interp_analysis.ipynb](4.2_Global_interp_analysis.ipynb), se ha observado que una misma regla puede combinar condiciones con significado conceptual para el dominio junto con otras destinadas principalmente a delimitar la región del espacio de características, sin aportar una interpretación urbanística directa. Esta dependencia jerárquica puede generar reglas con condiciones parcialmente redundantes o poco relevantes desde el punto de vista interpretativo frente a los mecanismos aditivos.

Un ejemplo de este fenómeno se observa en el grado de cerramiento *Individualista*. El árbol de decisión caracteriza esta categoría mediante cinco reglas construidas a partir de un conjunto amplio de variables predictoras, cuya interpretación depende de la partición inicial realizada por la variable *integración en el núcleo urbano (DIS_3)*. La capacidad discriminativa de esta variable resulta limitada en este grado de cerramiento, ya que las condiciones posteriores reflejan patrones similares independientemente de esta separación inicial. Asimismo, la aparición condicionada de variables como las *cámaras de seguridad (CSE)* en determinadas reglas dificulta su interpretación, al no existir una justificación conceptual clara para que su influencia dependa de las particiones previas del árbol. En consecuencia, la complejidad estructural y el número de reglas generadas limitan la obtención de una explicación compacta y coherente con el conocimiento del dominio.

En cambio, tanto la regresión logística como *XGBoost* mediante *SHAP*, identifican de forma consistente que los principales factores catalizadores de este grado de cerramiento son la *entrada por vivienda (PVI)* y la *entrada por bloque (PBL)*, mientras que las *cámaras de seguridad (CSE)* actúan como factor inhibidor de menor intensidad y la influencia de *DIS_3* es muy reducida. De este modo, el grado de cerramiento *Individualista* puede caracterizarse de forma más comprensible mediante la contribución aditiva del conjunto de variables más influyente para su clasificación.

Una de las ventajas que puede aportar un árbol de decisión es su capacidad para simplificar el espacio de características mediante una selección implícita de aquellas variables que proporcionan una mayor capacidad de particionamiento. Sin embargo, en este problema se ha observado que este comportamiento limita tanto la interpretabilidad como la capacidad de generalización del modelo. Esta situación puede observarse de forma clara en el bloque semántico ``*Seguridad y vigilancia*'', cuya influencia resulta especialmente relevante en la clasificación del grado de cerramiento *Protegido*.

En el árbol de decisión, únicamente la variable *cámaras de seguridad (CSE)* aparece en las reglas generadas, mientras que el resto de características asociadas a este bloque no intervienen en la decisión del modelo. Esta simplificación puede provocar que complejos residenciales que dispongan de otros servicios de vigilancia, pero no de cámaras de seguridad, sean clasificados incorrectamente al no quedar adecuadamente representados por las reglas aprendidas. Como consecuencia, la caracterización del grado de cerramiento *Protegido* resulta menos completa tanto desde el punto de vista predictivo como interpretativo.

Por el contrario, en la regresión logística y en *XGBoost*, todas las variables de este bloque contribuyen conjuntamente a la clasificación del grado de cerramiento *Protegido*. En particular, el análisis de *XGBoost* mediante *SHAP* permite diferenciar la influencia de los distintos servicios de seguridad. Las *cámaras de seguridad (CSE)* constituyen la característica con mayor impacto, cuya presencia favorece notablemente la pertenencia a esta categoría, mientras que su ausencia reduce dicha contribución en una magnitud similar. El resto de servicios presentan un efecto complementario: su presencia refuerza la clasificación como *Protegido*, aunque su ausencia tiene una influencia inhibitoria menor. Esta interpretación proporciona una caracterización más completa del fenómeno al identificar el servicio de seguridad dominante y los que funcionan de forma complementaria.

Por estos motivos, los mecanismos de interpretación aditivos se consideran los más adecuados para facilitar la transferencia de conocimiento a los expertos del dominio. En este contexto, la regresión logística multinomial se posiciona como el modelo transparente de referencia, al combinar un rendimiento superior al árbol de decisión con un mecanismo de explicabilidad intrínseca aditivo. Dentro de los modelos de caja negra, *XGBoost* se establece como el modelo candidato al obtener el rendimiento más elevado y estable en la evaluación realizada en [3.2_Performance_results.ipynb](3.2_Performance_results.ipynb), acompañado de un mecanismo de explicabilidad *post-hoc* de carácter aditivo como *SHAP*.

En términos de rendimiento global, la regresión logística multinomial obtiene resultados competitivos frente a *XGBoost*. No obstante, el análisis desagregado por grado de cerramiento revela diferencias entre ambos modelos. En particular, *XGBoost* proporciona una mejora notable en la precisión de la categoría *Simbólico*, junto con una mayor capacidad para identificar correctamente los complejos residenciales pertenecientes al resto de categorías. Esto resulta especialmente relevante en este problema, donde todos los grados de cerramiento tienen una importancia propia dentro del fenómeno estudiado. Además, la capacidad de *XGBoost* para capturar relaciones no lineales en los datos no solo mejora el rendimiento predictivo, sino que también enriquece la interpretación de los patrones aprendidos como se ha observado en la caracterización del grado de cerramiento *Simbólico*.

En consecuencia, se considera que la alternativa más adecuada para las características de este problema es *XGBoost* junto con el mecanismo de explicabilidad *SHAP*, al proporcionar el mayor rendimiento predictivo sin comprometer la interpretabilidad del modelo. Esta combinación permite conformar un sistema predictivo y explicable capaz de clasificar el grado de cerramiento de los complejos residenciales y explicar dicha clasificación de forma comprensible para los expertos del dominio.

Tanto la interpretabilidad basada en *SHAP* como la derivada del análisis de los coeficientes de la regresión logística representan enfoques conceptualmente similares, basados en la contribución aditiva de las variables, que se alinean de forma natural con la manera en que los expertos interpretan la relación entre las características de la morfología urbana y el grado de cerramiento. Esta similitud se refleja en las visualizaciones empleadas para la caracterización de los grados de cerramiento: los gráficos de barras asociados a los coeficientes de la regresión logística y los gráficos de barras y dispersión de *SHAP* utilizados para *XGBoost*. Por tanto, el uso de *SHAP* permite aprovechar la capacidad predictiva de un modelo de caja negra como *XGBoost* manteniendo un mecanismo de explicabilidad basado en principios interpretativos equivalentes a los de un modelo transparente como la regresión logística multinomial.